# Customer Churn Prediction — EDA & Modeling

This notebook explores the Telco-style customer churn dataset and prototypes the modeling approach that is productionized in `src/train_model.py`.

Run `python src/generate_sample_data.py` from the project root first if `data/raw/telco_churn.csv` does not exist yet.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_processing import load_raw_data, clean_data

sns.set_theme(style='whitegrid')
df = load_raw_data()
df.head()

## 1. Basic shape & missing values

In [ ]:
print(df.shape)
print(df.isnull().sum())
df.describe(include='all').T

## 2. Clean data (handles blank TotalCharges, encodes target)

In [ ]:
clean_df = clean_data(df)
clean_df['Churn'].value_counts(normalize=True)

## 3. Churn rate by key categorical drivers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ['Contract', 'InternetService', 'PaymentMethod']):
    rate = clean_df.groupby(col)['Churn'].mean().sort_values(ascending=False)
    sns.barplot(x=rate.index, y=rate.values, ax=ax)
    ax.set_title(f'Churn rate by {col}')
    ax.set_ylabel('Churn rate')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 4. Tenure & monthly charges vs churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=clean_df, x='tenure', hue='Churn', multiple='stack', ax=axes[0])
axes[0].set_title('Tenure distribution by churn')
sns.boxplot(data=clean_df, x='Churn', y='MonthlyCharges', ax=axes[1])
axes[1].set_title('Monthly charges by churn')
plt.tight_layout()
plt.show()

## 5. Model training & comparison

The full, production-ready training pipeline (with persistence of the model, preprocessor, metrics, and feature importances) lives in `src/train_model.py`. Run it from the project root with:

```bash
python -m src.train_model
```

Below we re-run the same comparison inline for exploration purposes.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from src.data_processing import build_preprocessor, get_feature_and_target
from src.train_model import get_candidate_models
from src import config

X, y = get_feature_and_target(clean_df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_SEED, stratify=y
)

preprocessor = build_preprocessor()
X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

for name, model in get_candidate_models().items():
    model.fit(X_train_t, y_train)
    proba = model.predict_proba(X_test_t)[:, 1]
    print(f'--- {name} ---')
    print('ROC-AUC:', round(roc_auc_score(y_test, proba), 4))
    print(classification_report(y_test, model.predict(X_test_t)))

## 6. Next steps

- Run `python -m src.train_model` to persist the best model to `models/churn_model.pkl`.
- Start the API: `uvicorn api.main:app --reload --port 8000`.
- Start the dashboard: `streamlit run frontend/streamlit_app.py`.